In [ ]:
import earthaccess
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from hmmlearn import hmm
from datetime import datetime

In [ ]:
auth = earthaccess.login()

In [ ]:
product = "GPM_3IMERGDF"  # GPM IMERG Final Daily
lat_bounds = (19.0, 19.6)  # Approximate CDMX
lon_bounds = (-99.4, -98.8)
start_date = "2015-01-01"
end_date = "2025-01-01"

In [ ]:
results = earthaccess.search_data(
    short_name=product,
    temporal=(start_date, end_date),
    bounding_box=(lon_bounds[0], lat_bounds[0], lon_bounds[1], lat_bounds[1])
)
print(f"Found {len(results)} files")

In [ ]:
files = earthaccess.download(results, "./gpm_data")

In [ ]:
ds = xr.open_mfdataset(files, engine="netcdf4")

In [ ]:
precip = ds["precipitationCal"].mean(dim=["lon", "lat"])
precip_series = precip.to_pandas()
precip_series.name = "precipitation"
precip_series = precip_series.dropna()

In [ ]:
def categorize_rain(value):
    if value < 0.1:
        return 0  # none
    elif value < 5:
        return 1  # low
    elif value < 20:
        return 2  # moderate
    else:
        return 3  # extreme

In [ ]:
model = hmm.MultinomialHMM(n_components=4, n_iter=200, random_state=42)

In [ ]:
obs = rain_states.values.reshape(-1, 1)
model.fit(obs)

In [ ]:
hidden_states = model.predict(obs)

print("Transition matrix (between hidden states):")
print(model.transmat_)

In [ ]:
last_state = hidden_states[-1]
next_day_prob = model.transmat_[last_state]
categories = ["none", "low", "moderate", "extreme"]

print("\nPredicted probabilities for next day rainfall:")
for cat, p in zip(categories, next_day_prob):
    print(f"{cat:>10}: {p:.3f}")

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(precip_series.index, precip_series.values, label="Rain (mm/day)", alpha=0.5)
plt.scatter(precip_series.index, rain_states, c=hidden_states, cmap="rainbow", s=5)
plt.title("Observed Rainfall and Hidden Markov States (CDMX)")
plt.xlabel("Date")
plt.ylabel("Rainfall (mm/day)")
plt.legend()
plt.show()